# 01 · Qubits, gates and entanglement / Qubit နှင့် gate များ

**Goal:** connect amplitudes, rotations and correlated measurements. About 15 minutes.
A pure-state amplitude vector has squared magnitudes summing to one. Measurement probabilities
do not reveal every complex phase in that vector.

Amplitude သည် complex number ဖြစ်နိုင်ပြီး probability သည် ၎င်း၏ absolute value squared ဖြစ်ပါသည်။


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__.split('.')[0] == '1', 'Install NexQus SDK 1.x in this kernel environment'
print('NexQus SDK:', nexqus.__version__)


In [ ]:
plus = Simulator().run(Circuit(1).h(0), shots=0)
print('Amplitudes:', plus.statevector)
print('Probabilities:', plus.probabilities)
np.testing.assert_allclose(np.abs(plus.statevector)**2, plus.probabilities)
assert np.isclose(np.vdot(plus.statevector, plus.statevector), 1)


RY(theta) on |0> gives probability of 1 equal to sin²(theta/2).
Angle များကို radians ဖြင့် သတ်မှတ်ပါ။ pi radians သည် 180 degrees ဖြစ်ပါသည်။


In [ ]:
angles = np.linspace(0, 2*np.pi, 41)
p_one = np.array([Simulator().run(Circuit(1).ry(float(t), 0), shots=0).probabilities[1] for t in angles])
np.testing.assert_allclose(p_one, np.sin(angles/2)**2, atol=1e-12)
plt.figure(figsize=(6, 3)); plt.plot(angles, p_one)
plt.xlabel('RY angle (radians)'); plt.ylabel('P(1)'); plt.title('A single-qubit rotation')
plt.tight_layout(); plt.show()


In [ ]:
bell = Simulator().run(algorithms.bell(), shots=0)
print('ZZ correlation:', bell.expectation('ZZ'))
print('XX correlation:', bell.expectation('XX'))
print('State of q0 alone:\n', bell.reduced_density([0]))
np.testing.assert_allclose(bell.reduced_density([0]), np.eye(2)/2, atol=1e-12)
assert np.isclose(bell.expectation('XX'), 1)
assert np.isclose(bell.expectation('ZZ'), 1)


Each Bell qubit alone is mixed even though the pair has a pure state. Correlation in both X and Z
is more informative than seeing matching computational-basis samples alone.

Entangled pair သည် pure state ဖြစ်သော်လည်း qubit တစ်ခုတည်းကို သီးခြားကြည့်လျှင် mixed state ဖြစ်နိုင်ပါသည်။


In [ ]:
# QFT is implemented from gates. Check inverse QFT on a complex input.
vector = np.array([1, 1j, 2, -1], complex); vector /= np.linalg.norm(vector)
round_trip = algorithms.qft(2).compose(algorithms.qft(2, inverse=True))
recovered = Simulator().run(round_trip, initial_state=vector, shots=0).statevector
np.testing.assert_allclose(recovered, vector, atol=1e-12)
grover = Simulator().run(algorithms.grover(2, '10'), shots=0)
assert grover.most_likely == '10'
assert np.isclose(grover.probabilities[2], 1)
print('QFT inverse and Grover checks passed.')


**Try it:** prepare `algorithms.ghz(3)` and predict its nonzero probabilities.
Simulator access to all amplitudes is not the same as measuring those amplitudes on hardware.
